# Task 1.1: Core Contribution / Architecture

## Paper: Learning Structural SVMs with Latent Variables
**Authors**: Chun-Nam Yu, Thorsten Joachims  
**Venue**: ICML 2009

---

## Step-by-Step Method Description

### Step 1: Formulate the Structural SVM with Latent Variables
- **Description**: The method extends standard Structural SVMs by introducing a latent variable $h$ that represents hidden structure not observed in the training data. The learning problem is formulated as:
  $$\min_{w} \frac{1}{2}||w||^2 + C \sum_{i=1}^{n} \xi_i$$
  subject to: $w^T \Psi(x_i, y_i, h_i^*) \geq w^T \Psi(x_i, \bar{y}, h') + \Delta(y_i, \bar{y}) - \xi_i$ for all $\bar{y} \neq y_i, h'$
- **Reference**: Equation (1) and Section 2 of the paper
- **Purpose**: This formulation allows the model to learn both the weight vector $w$ and infer the latent structure $h$ that best explains the observed input-output pairs, enabling the method to capture hidden patterns in structured prediction tasks.

### Step 2: Inference of Latent Variables (H-step in CCCP)
- **Description**: Given the current weight vector $w$, infer the most likely latent variable $h_i^*$ for each training example by solving:
  $$h_i^* = \arg\max_h w^T \Psi(x_i, y_i, h)$$
  This finds the hidden structure that maximizes the score function for the observed output $y_i$.
- **Reference**: Section 3, Algorithm 1 (CCCP procedure), and the H-step described around equation (3)
- **Purpose**: This step infers the latent states for each training sample given the current model parameters. It bridges the gap between the tractable convex-concave formulation and the non-convex original problem by committing to specific latent values.

### Step 3: Optimize Weights via Quadratic Program (W-step in CCCP)
- **Description**: With latent variables fixed, the problem becomes a standard Structural SVM that is convex in $w$. Solve:
  $$\min_{w} \frac{1}{2}||w||^2 + C \sum_{i=1}^{n} \xi_i$$
  subject to: $w^T \Psi(x_i, y_i, h_i^*) \geq w^T \Psi(x_i, \bar{y}, h_i^*) + \Delta(y_i, \bar{y}) - \xi_i$
  using a quadratic program solver (e.g., cutting-plane algorithm for Structural SVMs).
- **Reference**: Section 3, Algorithm 1, and equation (2) which shows the convex hull lower bound
- **Purpose**: This step optimizes the weight vector with respect to the fixed latent assignments, guaranteeing convergence to a local maximum of the original objective through the Concave-Convex Procedure (CCCP).

### Step 4: Iterate H-step and W-step Until Convergence
- **Description**: Alternate between inferring $h$ (Step 2) and optimizing $w$ (Step 3) until the objective value converges or reaches a specified number of iterations. The CCCP guarantees that the objective is non-decreasing at each iteration.
- **Reference**: Algorithm 1 in Section 3; the CCCP is a general technique proven to converge to a critical point
- **Purpose**: This iterative procedure solves the non-convex optimization problem by alternating between two tractable subproblems, each of which is easier to solve than the joint optimization over both $w$ and $h$.

### Step 5: Prediction on New Examples
- **Description**: For a test input $x_{test}$, jointly infer both the output $y$ and latent variable $h$ that maximize the score:
  $$(y^*, h^*) = \arg\max_{y,h} w^T \Psi(x_{test}, y, h)$$
  or for ranking tasks, compute scores and rank by $w^T \Psi(x, y, h)$ for latent rankingsof documents.
- **Reference**: Section 2 and application sections (Sections 4–6) describing predictions in motif finding, coreference, and ranking
- **Purpose**: This step produces final predictions on unseen data by jointly optimizing over both structured outputs and latent variables using the learned weights.

---

## Final Summary

This paper solves the problem of **learning discriminative models for structured prediction when hidden structure (latent variables) determines the correct output**, using a **Concave-Convex Procedure that guarantees convergence to a local maximum by alternating between latent variable inference and convex weight optimization**—an approach that extends Structural SVMs to handle latent variables without requiring explicit annotation of the hidden structure.